# Project: which model when?
**Machine Learning · Week 8 (The rest of the classical toolbox) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w8d5.html

Eight weeks, many models. Today they all compete: first on three small 2D problems, where you can *see* how
each one draws its boundary, then on real data, judged honestly with cross-validation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

models = {
    "k-nearest neighbors": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "naive Bayes": GaussianNB(),
    "decision tree": DecisionTreeClassifier(max_depth=5, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
    "SVM (RBF kernel)": make_pipeline(StandardScaler(), SVC()),
    "neural network": make_pipeline(StandardScaler(), MLPClassifier((32, 32), max_iter=2000, random_state=0)),
}
Xl, yl = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2, n_clusters_per_class=1,
                             class_sep=1.2, random_state=3)
datasets = {"moons": make_moons(n_samples=300, noise=0.3, random_state=0),
            "circles": make_circles(n_samples=300, noise=0.15, factor=0.5, random_state=1),
            "almost linear": (Xl, yl)}
apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])

fig, axes = plt.subplots(3, 8, figsize=(19, 7.4), layout="constrained")
for row, (dname, (X, y)) in enumerate(datasets.items()):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    x0, x1 = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y0, y1 = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x0, x1, 120), np.linspace(y0, y1, 120))
    for col, (mname, model) in enumerate(models.items()):
        ax = axes[row, col]
        model.fit(X_tr, y_tr)
        grid = np.c_[xx.ravel(), yy.ravel()]
        if hasattr(model, "predict_proba"):
            p = model.predict_proba(grid)[:, 1]
        else:                                                     # the SVM gives a score, not a probability
            p = 1 / (1 + np.exp(-model.decision_function(grid)))
        p = p.reshape(xx.shape)
        ax.contourf(xx, yy, p, levels=12, cmap=apricot_blue, alpha=0.6)
        ax.scatter(X_te[:, 0], X_te[:, 1], c=np.where(y_te == 0, "#ff8a50", "#5577f5"), s=7, edgecolors="none")
        ax.text(x1 - 0.1, y0 + 0.1, f"{model.score(X_te, y_te):.2f}", ha="right", va="bottom", fontsize=12, weight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(mname, fontsize=11)
        if col == 0:
            ax.set_ylabel(dname, fontsize=12)
plt.show()

## The real showdown: cross-validation on real data
Two real datasets: tumors (30 measurements each, two classes) and the 8 x 8 digits (64 pixels, ten classes).
Every model gets the same 5 folds. We compare the mean accuracy, how much it varies between folds, and the
training time. The test set stays locked away for the end.

In [ ]:
import time
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import cross_validate, StratifiedKFold

real = {"tumors": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
splits = {}
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for dname, (X, y) in real.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
    splits[dname] = (X_tr, X_te, y_tr, y_te)
    print(f"{dname}: {X.shape[0]} examples, {X.shape[1]} features, {len(set(y))} classes")
    print(f"  {'model':22s} {'CV accuracy':>13s}   {'fit time':>8s}")
    for mname, model in models.items():
        r = cross_validate(model, X_tr, y_tr, cv=folds)
        print(f"  {mname:22s} {r['test_score'].mean():.3f} ± {r['test_score'].std():.3f}   {1000 * r['fit_time'].mean():6.0f} ms")
    print()

## Tune the finalist, then open the test set once
`GridSearchCV` tries every combination of settings with cross-validation on the training set only, and refits
the best one. Then, and only then, one look at the test set.

In [ ]:
from sklearn.model_selection import GridSearchCV

X_tr, X_te, y_tr, y_te = splits["digits"]
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()),
                      {"svc__C": [1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1]}, cv=folds)
search.fit(X_tr, y_tr)
print("best settings:", search.best_params_)
print(f"best CV accuracy:  {search.best_score_:.3f}")
print(f"final test accuracy (looked at once): {search.score(X_te, y_te):.3f}")

## Your turn: a third dataset
The wine dataset has 13 chemical measurements of 178 wines from three growers. Split off a test set, run
every model through 5-fold cross-validation on the training part, pick the winner, and report its test
accuracy once.

In [ ]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
scores = {}
for mname, model in models.items():
    scores[mname] = None    # TODO: the mean of cross_validate(model, X_tr, y_tr, cv=folds)["test_score"]
winner = None               # TODO: the name with the highest score
print(scores)
# TODO: fit models[winner] on all of X_tr, y_tr and print its accuracy on X_te, y_te

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
scores = {mname: cross_validate(model, X_tr, y_tr, cv=folds)["test_score"].mean() for mname, model in models.items()}
for mname, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"{mname:22s} {s:.3f}")
winner = max(scores, key=scores.get)
test = models[winner].fit(X_tr, y_tr).score(X_te, y_te)
print(f"\nwinner: {winner}, test accuracy {test:.3f}")
print("Three models tie at the top and all eight are within about 2.5 points: with 133 training wines, that is")
print("three or four wines. A tie is a tie, so pick the simplest of the leaders (here, logistic regression).")
print("And a perfect score on 45 test wines is partly luck: don't expect 100% on the next 45.")